# Merge gsd files

In [3]:
with (open('ru_gsd-ud-dev.conllu.txt', encoding='utf-8') as f1, 
      open('ru_gsd-ud-test.conllu.txt', encoding='utf-8') as f2, 
      open('ru_gsd-ud-train.conllu.txt', encoding='utf-8') as f3,
      open('ru_gsd_merged.txt', 'w', encoding='utf-8') as merged):
    merged.write(f'{f1.read()}{f2.read()}{f3.read()}')

# Corpus manager

In [ ]:
from random import random

In [1]:
# # sent_id = dev-s21
# # text = Вышивки Полесья -- простые и четкие по композиции.
# 1	Вышивки	вышивка	NOUN	NN	Animacy=Inan|Case=Nom|Gender=Fem|Number=Plur	4	nsubj	_	_
# 2	Полесья	Полесье	PROPN	NNP	Animacy=Inan|Case=Gen|Gender=Neut|Number=Sing	1	nmod	_	_
# 3	--	--	PUNCT	--	_	4	punct	_	_
# 4	простые	простой	ADJ	JJL	Case=Nom|Degree=Pos|Number=Plur	0	root	_	_
# 5	и	и	CCONJ	CC	_	6	cc	_	_
# 6	четкие	четкий	ADJ	JJL	Case=Nom|Degree=Pos|Number=Plur	4	conj	_	_
# 7	по	по	ADP	IN	_	8	case	_	_
# 8	композиции	композиция	NOUN	NN	Animacy=Inan|Case=Dat|Gender=Fem|Number=Sing	6	nmod	_	SpaceAfter=No
# 9	.	.	PUNCT	.	_	4	punct	_	_

In [2]:
class Token:
    def __init__(self, token_info):
        self.id, self.form, self.lemma, self.pos, self.xpos, self.gr, self.head, self.deprel, self.deps, self.misc = token_info.split('\t')
        self.id = int(self.id)
        if self.gr != '_':
            for cath_value in self.gr.split('|'):
                cathegory, value = cath_value.split('=')
                setattr(self, cathegory.lower(), value)
            
    def __str__(self):
        return '\t'.join(f'{k}:{v}' for k, v in self.__dict__.items() if not k.startswith('__') and not callable(k))
    
    def __repr__(self):
        return '\t'.join(f'{k}:{v}' for k, v in self.__dict__.items() if not k.startswith('__') and not callable(k))

In [10]:
class Sentence:
    def __init__(self, sentence_info):
        self.sent_id, self.sentence_text, *self.tokens = sentence_info.split('\n')
        self.sentence_text = self.sentence_text.replace('# text = ', '')
        self.tokens = [Token(token_info=token) for token in self.tokens]
    
    def __str__(self):
        return f'{self.sent_id}\n{self.sentence_text}\n'+'\n'.join(str(token) for token in self.tokens)
    
    def __repr__(self):
        return f'{self.sent_id}\n{self.sentence_text}\n'+'\n'.join(str(token) for token in self.tokens)
    
    def _search_by_token(self, query_token:str, kwic_len:int):
        if query_token in self.sentence_text:
            for tk in self.tokens:
                if tk.form == query_token:
                    query_token_index = tk.id
                    kwik = [tk for tk in self.tokens if abs(tk.id-query_token_index) <= kwic_len]
                    return kwik
        
    def _search_by_lemma(self, query_lemma:str, kwic_len:int):
        for tk in self.tokens:
            if tk.lemma == query_lemma:
                query_token_index = tk.id
                kwik = [tk for tk in self.tokens if abs(tk.id-query_token_index) <= kwic_len]
                return kwik
                
    def _general_search(self, kwic_len:int, token=None, lemma=None, pos=None, xpos=None, gr=None, deprel=None, **kwargs):
        query = {'form': token, 'lemma':lemma, 'pos': pos, 'xpos': xpos, 'gr': gr, 'deprel': deprel}
        query = {k:v for k, v in query.items() if v is not None}
        if kwargs:
            for k, v in kwargs.items():
                query[k] = v
        
        for tk in self.tokens:
            condition = all(tk.__getattribute__(key)==value for key, value in query.items())
            if condition:
                query_token_index = tk.id
                kwik = [tk for tk in self.tokens if abs(tk.id-query_token_index) <= kwic_len]
                return kwik

In [11]:
class Corpus:
    def __init__(self):
        self.sentences = []

    def load_from_file(self, filepath):
        with open(filepath, encoding='utf-8') as corpus_file:
            self.sentences = [Sentence(sent) for sent in corpus_file.read().split('\n\n') if sent]

    def search_by_token(self, token, n_examples=5, kwic_len=5):
        qwery_answer = []
        for sentence in sorted(self.sentences, key=random()):
            if len(qwery_answer) == n_examples:
                return qwery_answer
            c = sentence._search_by_token(query_token=token, kwic_len=kwic_len)
            if c:
                qwery_answer.append(c)
        if qwery_answer:
            return qwery_answer
        return f'Примеров для {token=} в корпусе не нашлось.'
    
    def search_by_lemma(self, lemma, n_examples=5, kwic_len=5):
        qwery_answer = []
        for sentence in sorted(self.sentences, key=random()):
            if len(qwery_answer) == n_examples:
                return qwery_answer
            c = sentence._search_by_lemma(query_lemma=lemma, kwic_len=kwic_len)
            if c:
                qwery_answer.append(c)
        if qwery_answer:
            return qwery_answer
        return f'Примеров для {lemma=} в корпусе не нашлось.'

    def general_search(self, token=None, lemma=None, pos=None, xpos=None, gr=None, deprel=None, n_examples=5, kwic_len=5, **kwargs):
        qwery_answer = []
        for sentence in sorted(self.sentences, key=random()):
            if len(qwery_answer) == n_examples:
                return qwery_answer
            c = sentence._general_search(token=token, lemma=lemma, pos=pos, xpos=xpos, gr=gr, deprel=deprel, kwic_len=kwic_len, **kwargs)
            if c:
                qwery_answer.append(c)
        if qwery_answer:
            return qwery_answer
        plug = ' '.join([f'{x=}' for x in [lemma, pos, xpos, gr, deprel] if x is not None])
        return f'Примеров для {plug} в корпусе не нашлось.'

In [18]:
CORPUS = Corpus()
CORPUS.load_from_file('ru_gsd_merged.txt')

In [20]:
def simple_kwic_output(kwics):
    if isinstance(kwics[0], list):
        return "\n".join(' '.join([tk.form for tk in kwic]).replace(' ,', ',').replace(' .', '.') for kwic in kwics)
    elif isinstance(kwics, str):
        return kwics
    else:
        return ' '.join([tk.form for tk in kwics])

In [21]:
print(simple_kwic_output(CORPUS.general_search(lemma='год', kwic_len=5, number='Plur')))

В течение 3 лет 10 месяцев Виноградов являлся младшим
её месте выстроен в 80-х годах новый мемориал павших при освобождении
приезжал в Дубно в 1960-х годах и нашел дом в Дубно
Сатпаев : В последние несколько лет со стороны многих наших чиновников
Крымской войны 1853 -- 1856 годов, в Турцию.
